# PAISA optional OGDC LSTM experiment

This optional Colab workflow uses the repository's reusable LSTM trainer. The trainer owns chronological sequence construction, train-only scaling, metrics, prediction validation, and immutable artifact registration; the notebook is only orchestration and presentation. TensorFlow remains outside the default CPU dependencies.

The output is an academic experiment and is not financial advice.

## 1. Resolve PAISA and install optional TensorFlow

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Arfa01/PAISA.git"
GIT_REF = os.environ.get("PAISA_GIT_REF", "main")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_DIR = Path("/content/PAISA")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif not (REPO_DIR / "pyproject.toml").is_file():
        raise RuntimeError(f"Existing path is not a PAISA checkout: {REPO_DIR}")
    origin = subprocess.run(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR,
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing checkout has unexpected origin: {origin}")
    dirty = subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_DIR,
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("Existing Colab checkout is dirty; preserve or remove it before continuing.")
    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    current = Path.cwd()
    REPO_DIR = current if (current / "pyproject.toml").is_file() else current / "PAISA"
    if not (REPO_DIR / "pyproject.toml").is_file():
        raise RuntimeError(f"Run the notebook from a PAISA checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow>=2.16,<3.0"], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
print("Repository:", REPO_DIR)
print("Resolved commit:", commit)
print("TensorFlow is notebook-local and is not required by the CPU trainer.")

## 2. Fetch or load OGDC and call shared preparation

Set `PAISA_PRICES_PATH` to a cleaned real-data CSV to avoid a live request. Otherwise the existing PSX pipeline writes into an isolated temporary directory so canonical v0 files are not replaced.

In [ ]:
from datetime import date
import tempfile

import pandas as pd

from paisa.experiments.trainer import prepare_training_data
from paisa.pipeline import run_v0_pipeline

prices_value = os.environ.get("PAISA_PRICES_PATH")
if prices_value:
    prices = pd.read_csv(Path(prices_value))
    provider_name = "input-csv"
else:
    acquisition_dir = Path(tempfile.mkdtemp(prefix="paisa-lstm-provider-"))
    metadata = run_v0_pipeline(
        symbols=["OGDC"], provider_name="psx-dps", start=date(2021, 1, 1),
        train=False, data_dir=acquisition_dir / "data", model_dir=acquisition_dir / "models",
    )
    prices = pd.read_csv(metadata["files"]["prices"])
    provider_name = str(metadata["provider"])

prepared = prepare_training_data(prices, symbol="OGDC")
print("Dataset SHA-256:", prepared.dataset_sha256)
print("Ordered features:", list(prepared.feature_columns))
print({name: len(getattr(prepared.splits, name)) for name in ("train", "validation", "test")})

## 3. Train and register the immutable inference bundle

The reusable trainer fits `StandardScaler` on training rows only, uses earlier rows as context without moving their targets across splits, disables shuffling, validates every prediction, and stores the Keras model bytes plus fitted scaler in one checksum-protected registry artifact.

In [ ]:
from paisa.experiments.lstm import LSTMTrainingConfig, train_and_register_lstm

config = LSTMTrainingConfig(
    sequence_length=20, units=32, dropout=0.20, learning_rate=0.001,
    epochs=50, batch_size=32, patience=6, random_seed=42, verbose=1,
)
result = train_and_register_lstm(
    prepared=prepared,
    models_root=Path("artifacts/lstm"),
    repo_root=REPO_DIR,
    config=config,
    provider=provider_name,
)
print("Registered model ID:", result.model_id)
print("Artifact directory:", result.model_dir)
print("Artifact checksum:", result.manifest["artifact_sha256"])

## 4. Validate and display saved evidence

In [ ]:
from IPython.display import display

from paisa.experiments.artifacts import make_zip_from_directory, validate_model_manifest
from paisa.experiments.evaluation import validate_prediction_frame

validate_model_manifest(result.manifest)
predictions = validate_prediction_frame(pd.read_csv(result.model_dir / "predictions.csv"))
metric_rows = pd.DataFrame(
    [{"split": split_name, **values} for split_name, values in result.metrics.items()]
).drop(columns=["confusion_matrix", "class_counts", "predicted_class_counts"], errors="ignore")
display(metric_rows)
display(predictions.head())
zip_path = make_zip_from_directory(
    result.model_dir, result.model_dir.parent / f"{result.model_id}.zip"
)
print("Validated manifest:", result.model_dir / "manifest.json")
print("Backtest-ready predictions:", result.model_dir / "predictions.csv")
print("ZIP:", zip_path)

## 5. Plot learning curves and optionally download

In [ ]:
import matplotlib.pyplot as plt

history_frame = pd.DataFrame(result.history)
history_frame[["loss", "val_loss"]].plot(title="LSTM loss")
plt.show()
history_frame[["accuracy", "val_accuracy"]].plot(title="LSTM accuracy")
plt.show()

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))